# Marketing Science Paper — `jaredpalmer/kev-9b` Training Suite

Trains the beta-loss paper's fine-tuned rows (2, 2 Ablation, 4, 4 Ablation) on **Kev-9B** (`LoRA rank = 64`, `bf16`, `S = 40`) under the unified semi-supervised objective in `paper_beta`:

$$\mathcal{L}(\theta) = \mathbb{E}_{d \in \mathcal{D}_{\text{CE}}}\!\big[\mathcal{L}_{\text{CE}}(y_d, p_\theta)\big] + \lambda\,\mathbb{E}_{d \in \mathcal{D}_{\beta}}\!\big[\widehat{\mathcal{L}}_{\text{pop-}\beta}^{\text{split}}(p_\theta, \beta^*_d)\big] + \alpha\,\mathbb{E}_{d \in \mathcal{D}_{\beta}}\!\big[\mathrm{KL}(p_0 \,\|\, p_\theta)\big]$$

| Row | Folder Slug | Respondent Choices ($\mathcal{D}_{\text{CE}}$) | Fitted OLS Table ($\mathcal{D}_{\beta}$) | $\lambda$ | $\alpha$ | Peak LR | Epochs | Steps |
|---|---|---|---|---:|---:|---:|---:|---:|
| **Row 1** | *(cached step 0)* | None | None | — | — | — | 0 | 0 |
| **Row 2** | `row2_wu_table_anchored` | None | `wu2025` (8 slopes) | 1.0 | 0.005 | `5e-6` | 3 | 1,395 |
| **Row 2 (Ablation)** | `row2_ablation_wu_table_no_kl` | None | `wu2025` (8 slopes) | 1.0 | 0.0 | `5e-6` | 3 | 1,395 |
| **Row 4** | `row4_hybrid_berke_ce_wu_table_anchored` | `berke2023` (36,297 choices) | `wu2025` (8 slopes, $y$ withheld) | 1.0 | 0.005 | `5e-6` | 3 | 4,125 |
| **Row 4 (Ablation)** | `row4_hybrid_berke_ce_wu_table_no_kl` | `berke2023` (36,297 choices) | `wu2025` (8 slopes, $y$ withheld) | 1.0 | 0.0 | `5e-6` | 3 | 4,125 |

Rows 2b, 3 and 5 (cross-entropy on respondent choices only) belong to the sim-respondents paper: they are trained by its notebooks in `code/papers/sim-respondents/train/`, and this paper reads their runs from there.

**Checkpoint selection (`BEST_METRIC = "kriner2026/beta_mse"`)**: OLS on the model's predicted choices for Kriner 2026's respondents and tasks, compared with the 12 OLS slopes fitted on their choices. Kriner's choices $y$ never enter training, but validation uses its respondents' personas and task designs. `grummon2025` is used for neither training nor checkpoint selection.

In [ ]:
import os
import sys
import time
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display

pd.set_option("display.max_columns", 24)
pd.set_option("display.width", 180)

# Walk up from the notebook folder to locate the repository's `code/` root
CODE_DIR = next(
    (c for p in (Path.cwd(), *Path.cwd().parents) for c in (p, p / "code") if all((c / d).is_dir() for d in ("kev", "data", "models"))),
    None,
)
if CODE_DIR is None:
    raise RuntimeError(f"No code/ root containing kev/, data/, and models/ at or above {Path.cwd()}")

PAPER_DIR = CODE_DIR / "papers" / "beta-loss"
SRC_DIR = PAPER_DIR / "src"
if not (SRC_DIR / "paper_beta").is_dir():
    raise RuntimeError(f"`paper_beta` package not found in {SRC_DIR}")

# Run from `code/` so data loaders and shared disk caches resolve identically to all beta-training notebooks
os.chdir(CODE_DIR)
for _p in (str(CODE_DIR), str(CODE_DIR / "kev"), str(SRC_DIR)):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from models.kev_pointer import KevPointerModel
from training import free_gpu

import beta_ols
from beta_ols import core as beta_core
from beta_train.diagnostics import find_max_micro_chunk
from beta_train.items import build_items_cached, items_summary
from beta_train.model_state import set_checkpointing, snapshot_trainable
from beta_train.validation import PopulationValidator, validate_all_cached, validation_metrics_frame
from paper_beta import (
    PaperTrainConfig,
    run_paper_training,
    summarize_paper_runs_dir,
)
from paper_beta.trainer import paper_steps_per_epoch

# Shared disk cache for tokenized items and untrained step-0 validation
CACHE_DIR = PAPER_DIR / ".cache" / "beta_training"
MODEL_SLUG = "kev-9b"
RUNS_DIR = PAPER_DIR / "runs" / MODEL_SLUG

print(f"CODE_DIR : {CODE_DIR}")
print(f"PAPER_DIR: {PAPER_DIR}")
print(f"RUNS_DIR : {RUNS_DIR}")

In [ ]:
# Require CUDA and release any previously loaded `kev_model` before allocating a fresh 9B backbone
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required to train jaredpalmer/kev-9b in bf16")
if "kev_model" in globals():
    globals().pop("kev_model")
device = "cuda"
free_gpu()
torch.manual_seed(42)

KEV_BASE = "jaredpalmer/kev-9b@db029f08b290afd9fee4aa4bbcd9ae48602d1eb0"
LORA_RANK = 64
WEIGHTS_DTYPE = "bf16"
HEAD_TEMPERATURE = 1.0
GRADIENT_CHECKPOINTING = False

t_model0 = time.perf_counter()
kev_model = KevPointerModel.build(
    KEV_BASE,
    device=device,
    lora_rank=LORA_RANK,
    weights_dtype=WEIGHTS_DTYPE,
)
kev_model.net.head.temperature = HEAD_TEMPERATURE
kev_model.train()
set_checkpointing(kev_model, GRADIENT_CHECKPOINTING)
torch.cuda.synchronize()
model_load_sec = time.perf_counter() - t_model0

trainable_params_count = sum(p.numel() for p in kev_model.trainable_parameters())
total_params_count = sum(p.numel() for p in kev_model.parameters())
display(
    pd.Series(
        {
            "base_model": KEV_BASE,
            "mode": kev_model.mode,
            "training": kev_model.training,
            "weights_dtype": kev_model.meta.weights_dtype,
            "lora_rank": kev_model.meta.lora,
            "trainable_params": f"{trainable_params_count:,}",
            "total_params": f"{total_params_count:,}",
            "gradient_checkpointing": GRADIENT_CHECKPOINTING,
            "model_load_sec": round(model_load_sec, 3),
        },
        name="kev_9b_train_mode",
    ).to_frame()
)

INIT_TRAINABLE_STATE = snapshot_trainable(kev_model)
MODEL_KEY = {
    "base": KEV_BASE,
    "lora_rank": LORA_RANK,
    "weights_dtype": WEIGHTS_DTYPE,
    "head_temperature": HEAD_TEMPERATURE,
}
print(f"Snapshotted {len(INIT_TRAINABLE_STATE)} trainable tensors")

# 1. Datasets & Fitted OLS Targets (`wu2025` Table)

In [ ]:
ITEM_DATASETS = ["wu2025", "berke2023"]
VALIDATION_DATASETS = ["kriner2026", "grummon2025"]
LOADED_DATASETS = [*ITEM_DATASETS, *VALIDATION_DATASETS]

choice_dfs: dict[str, pd.DataFrame] = {}
regression_rows_by_dataset: dict[str, pd.DataFrame] = {}
regression_specs_by_dataset: dict[str, dict] = {}
human_ols_by_dataset: dict[str, pd.DataFrame] = {}
module_by_dataset = {}
piece_names_by_study: dict[str, tuple[str, ...]] = {}

for dataset_name in LOADED_DATASETS:
    dataset_module = beta_ols.DATASET_MODULES[dataset_name]
    full_choice_df = dataset_module.load_choice_df()
    pieces = {}
    if dataset_name in VALIDATION_DATASETS:
        pieces[dataset_name] = full_choice_df
    if dataset_name in ITEM_DATASETS:
        task_pieces = beta_core.split_by_task(full_choice_df, dataset_name)
        pieces.update(task_pieces)
        piece_names_by_study[dataset_name] = tuple(task_pieces)

    for piece_name, choice_df in pieces.items():
        rows = dataset_module.regression_rows(choice_df)
        specs = dataset_module.regression_specs(rows)
        module_by_dataset[piece_name] = dataset_module
        choice_dfs[piece_name] = choice_df
        regression_rows_by_dataset[piece_name] = rows
        regression_specs_by_dataset[piece_name] = specs
        human_ols_by_dataset[piece_name] = beta_core.fit_human_ols(rows, specs)
        print(
            f"{piece_name}: {choice_df['CASEID'].nunique():,} respondents | "
            f"{len(choice_df):,} tasks | {len(specs)} regression(s)"
        )
        display(beta_core.wide_ols_table(human_ols_by_dataset[piece_name]).round(4))

WU_DATASETS = piece_names_by_study["wu2025"]
BERKE_DATASETS = piece_names_by_study["berke2023"]
ALL_TRAIN_DATASETS = WU_DATASETS + BERKE_DATASETS

# Table targets only for `wu2025`: the 8 standardized OLS slopes fitted on Wu's choices, standing in for a published table
# (Wu's y never enters the loss in Row 2 or Row 4)
target_std_betas_by_dataset: dict[str, dict[str, torch.Tensor]] = {}
for dataset_name in WU_DATASETS:
    dataset_ols = human_ols_by_dataset[dataset_name]
    dataset_targets = {}
    for group, spec in regression_specs_by_dataset[dataset_name].items():
        group_ols = dataset_ols[dataset_ols["model"] == group].set_index("term")
        target_values = group_ols.loc[spec.target_labels, "std_beta"].to_numpy(dtype="float64")
        dataset_targets[group] = torch.tensor(target_values, dtype=torch.float64)
        display(
            pd.Series(
                target_values,
                index=spec.target_labels,
                name=f"{dataset_name} | {group} fitted OLS target std_beta",
            ).round(4).to_frame()
        )
    target_std_betas_by_dataset[dataset_name] = dataset_targets

print(f"WU_DATASETS    = {WU_DATASETS}")
print(f"BERKE_DATASETS = {BERKE_DATASETS}")

# 2. Held-Out Validators, Row 1 (Untrained Base LLM), & Cached Training Items

In [ ]:
VALIDATION_MAX_QUESTIONS = 1024
VALIDATION_MAX_CHARS = 1_000_000

validators = {
    dataset_name: PopulationValidator(
        beta_ols.DATASET_MODULES[dataset_name],
        choice_dfs[dataset_name],
        regression_specs_by_dataset[dataset_name],
        max_questions=VALIDATION_MAX_QUESTIONS,
        max_chars=VALIDATION_MAX_CHARS,
    )
    for dataset_name in VALIDATION_DATASETS
}
display(pd.DataFrame([v.describe() for v in validators.values()]).set_index("validator"))

In [ ]:
# Row 1 of the paper table: Untrained Base LLM (step-0 evaluation, cached in this paper's .cache/beta_training)
step0_validation, step0_cache_report = validate_all_cached(
    kev_model,
    validators,
    INIT_TRAINABLE_STATE,
    model_key=MODEL_KEY,
    cache_dir=CACHE_DIR,
)
display(step0_cache_report)
display(validation_metrics_frame(step0_validation).round(5))
for dataset_name, (_, beta_df) in step0_validation.items():
    display(beta_df.round(4))

In [ ]:
# Tokenized training items for Wu + Berke (reuses this paper's .cache/beta_training)
item_modules = {dataset_name: module_by_dataset[dataset_name] for dataset_name in ALL_TRAIN_DATASETS}
unified_items, items_cache_report = build_items_cached(
    kev_model, item_modules, choice_dfs, cache_dir=CACHE_DIR
)
display(items_cache_report)
display(items_summary(unified_items, regression_specs_by_dataset))

# 3. Micro-Chunk Sizing

`RUN_MICRO_CHUNK_SEARCH = True`: the largest micro-chunk that fits for Kev-9B across `wu2025` and `berke2023` with `GRADIENT_CHECKPOINTING = False` is measured on this GPU, up to `max_chunk = 41` (one chunk holds a whole `S = 40` or `41` sample).

In [ ]:
MAX_FITTING_MICRO_CHUNK = 1
RUN_MICRO_CHUNK_SEARCH = True

if RUN_MICRO_CHUNK_SEARCH:
    search_items = [item for ds in ALL_TRAIN_DATASETS for item in unified_items[ds]]
    micro_chunk_search_df, MAX_FITTING_MICRO_CHUNK = find_max_micro_chunk(
        kev_model,
        search_items,
        INIT_TRAINABLE_STATE,
        max_chunk=41,
        gradient_checkpointing=GRADIENT_CHECKPOINTING,
        candidates_per_size=3,
    )
    if MAX_FITTING_MICRO_CHUNK>1: MAX_FITTING_MICRO_CHUNK-=1 # for safety decrease microchunk by 1
    display(micro_chunk_search_df)
print(f"Using MAX_FITTING_MICRO_CHUNK = {MAX_FITTING_MICRO_CHUNK} (gradient_checkpointing={GRADIENT_CHECKPOINTING})")

# 4. Shared Training Protocol & Per-Row Runners

In [ ]:
NOTEBOOK = "02-train-paper-9b"
# Checkpoint selection for every row: Kriner 2026 beta MSE (model OLS on Kriner's respondents vs the 12 OLS slopes fitted on their choices)
BEST_METRIC = "kriner2026/beta_mse"
BEST_MODE = "min"

from_scratch = False

SAMPLE_SIZE = 40
# One peak learning rate for every regime (OneCycle max)
PEAK_LR = 5e-6
EVAL_EVERY_ROWS_BY_VALIDATOR = {
    "kriner2026": 1280,   # every 32 steps
    "grummon2025": 6400,  # every 160 steps (plus step 0, final step, and held-out best.pt)
}
EVAL_EVERY_BY_VALIDATOR = {
    name: max(1, round(rows / SAMPLE_SIZE)) for name, rows in EVAL_EVERY_ROWS_BY_VALIDATOR.items()
}
MICRO_CHUNK = MAX_FITTING_MICRO_CHUNK
P0_CHUNK = 32
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
SEED = 42

RUN_METADATA = {
    "notebook": NOTEBOOK,
    "model": MODEL_KEY,
    "item_datasets": ITEM_DATASETS,
    "validation_datasets": VALIDATION_DATASETS,
    "validator_kind": "fused",
    "eval_every_rows_by_validator": EVAL_EVERY_ROWS_BY_VALIDATOR,
}
runs = {}


# Trains one regime, declared in its own cell below: which datasets train on their choices (CE), which on their table
# (population beta loss, weight beta_lambda, plus KL to the base model, weight kl_anchor_alpha), and how many epochs.
# `slug` is the run folder name; `row_code` is the paper row, saved in run.json and used to order the summary
def train_paper_regime(
    *,
    slug: str,
    row_code: str,
    ce_datasets: tuple[str, ...],
    beta_datasets: tuple[str, ...],
    beta_lambda: float,
    kl_anchor_alpha: float,
    num_epochs: int,
):
    sample_size_by_dataset = {}
    for dataset_name in ce_datasets + beta_datasets:
        sample_size_by_dataset[dataset_name] = SAMPLE_SIZE
    run_config = PaperTrainConfig(
        ce_datasets=ce_datasets,
        beta_datasets=beta_datasets,
        sample_size_by_dataset=sample_size_by_dataset,
        beta_lambda=beta_lambda,
        kl_anchor_alpha=kl_anchor_alpha,
        lr=PEAK_LR,
        num_epochs=num_epochs,
        eval_every_by_validator=EVAL_EVERY_BY_VALIDATOR,
        lr_schedule="onecycle",
        weight_decay=WEIGHT_DECAY,
        micro_chunk=MICRO_CHUNK,
        p0_chunk=P0_CHUNK,
        gradient_checkpointing=GRADIENT_CHECKPOINTING,
        grad_clip=GRAD_CLIP,
        seed=SEED,
    )
    epoch_steps = paper_steps_per_epoch(unified_items, run_config)
    print(f"{row_code} — {slug}: {epoch_steps} steps per epoch, {epoch_steps * num_epochs} steps")
    display(pd.Series(run_config.as_dict(), name=slug).to_frame())
    runs[slug] = run_paper_training(
        model=kev_model,
        items_by_dataset=unified_items,
        specs_by_dataset=regression_specs_by_dataset,
        target_std_betas_by_dataset=target_std_betas_by_dataset,
        validators=validators,
        step0_validation=step0_validation,
        init_snapshot=INIT_TRAINABLE_STATE,
        config=run_config,
        device=device,
        run_dir=RUNS_DIR / slug,
        best_metric=BEST_METRIC,
        best_mode=BEST_MODE,
        run_metadata={**RUN_METADATA, "paper_row": row_code, "regime_slug": slug},
        from_scratch=from_scratch,
    )
    return runs[slug]

## Row 2 — Scenario A (No Microdata): Table Only (`Wu` fitted OLS $\beta^*$ + KL Anchor $\alpha = 0.005$)

Trains only on Wu 2025's **8 fitted OLS slopes**, standing in for a published table (`wu2025` respondent choices $y$ never enter the loss or gradient), at `lr = 5e-6`, `3` epochs, plus forward $\mathrm{KL}(p_0 \,\|\, p_\theta)$ toward the base model ($\alpha = 0.005$). The 8 slopes constrain only 8 directions of the model's probabilities on Wu's 18,591 tasks; the KL term pulls the rest toward the base model's predictions, and the Row 2 ablation tests whether that helps.

In [ ]:
train_paper_regime(
    slug="row2_wu_table_anchored",
    row_code="Row 2",
    ce_datasets=(),
    beta_datasets=WU_DATASETS,
    beta_lambda=1.0,
    kl_anchor_alpha=0.005,
    num_epochs=3,
)

## Row 2 Ablation — Table Only Without KL Anchor ($\alpha = 0.0$)

Identical to Row 2 (`lr = 5e-6`, `3` epochs) with $\alpha = 0.0$, to test whether the KL anchor changes the held-out results on Kev-9B (`kriner2026`, `grummon2025`).

In [ ]:
train_paper_regime(
    slug="row2_ablation_wu_table_no_kl",
    row_code="Row 2 (Ablation)",
    ce_datasets=(),
    beta_datasets=WU_DATASETS,
    beta_lambda=1.0,
    kl_anchor_alpha=0.0,
    num_epochs=3,
)

## Row 4 — Scenario B Proposed Hybrid: In-House `Berke` $y$ CE + External `Wu` Table (fitted OLS $\beta^*$)

Jointly interleaves `berke2023` respondent choices (1-pass micro-chunked CE) with `wu2025`'s 8 fitted OLS slopes (2-pass split population-$\beta$ cross loss + $\alpha = 0.005$ KL anchor; `wu2025` respondent choices $y$ strictly withheld).

In [ ]:
train_paper_regime(
    slug="row4_hybrid_berke_ce_wu_table_anchored",
    row_code="Row 4",
    ce_datasets=BERKE_DATASETS,
    beta_datasets=WU_DATASETS,
    beta_lambda=1.0,
    kl_anchor_alpha=0.005,
    num_epochs=3,
)

## Row 4 Ablation — Hybrid Without KL Anchor ($\alpha = 0.0$)

Identical to Row 4 with $\alpha = 0.0$.


In [ ]:
train_paper_regime(
    slug="row4_hybrid_berke_ce_wu_table_no_kl",
    row_code="Row 4 (Ablation)",
    ce_datasets=BERKE_DATASETS,
    beta_datasets=WU_DATASETS,
    beta_lambda=1.0,
    kl_anchor_alpha=0.0,
    num_epochs=3,
)

# 5. Summary of Saved `kev-9b` Paper Runs

Reads directly from `runs/kev-9b/` in the paper folder (needs only Cell 1 imports, so it can be run without loading the GPU model).

In [ ]:
SUMMARY_VALIDATORS = ("kriner2026", "grummon2025")
SUMMARY_METRICS = ("beta_mse", "std_beta_mse", "beta_mae", "spearman_rho", "sign_acc", "kl_loss")
display(summarize_paper_runs_dir(RUNS_DIR, SUMMARY_VALIDATORS, SUMMARY_METRICS).round(5))